In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Upload dataset zip ke Colab
# Cara 1: Upload manual (kalau dataset kecil)
from google.colab import files
print("Upload file dataset.zip kamu:")
uploaded = files.upload()

# Extract dataset
import zipfile
import os

# Unzip dataset
for filename in uploaded.keys():
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('/content/dataset')
        
print("Dataset extracted!")

In [ ]:
# ATAU kalau dataset udah di Google Drive
# Uncomment baris ini dan sesuaikan path
!unzip "/content/drive/MyDrive/path/to/your/dataset.zip" -d /content/dataset

In [2]:
# Import semua library yang dibutuhkan
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
import os
import random
import time
from tqdm import tqdm


# biar gak muncul warning teruswarnings.filterwarnings('ignore')
import warnings

In [ ]:
# fungsi buat set random seed biar hasil reproducible
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)  # pake 42 aja, angka favorit

In [ ]:
# Setup path ke dataset
import os

# Cek environment (local vs Colab)
try:
    import google.colab
    IN_COLAB = True
    # Path di Colab setelah extract
    dataset_dir = '/content/dataset'
except:
    IN_COLAB = False
    # Path lokal
    current_dir = os.path.dirname(os.path.abspath('main.ipynb'))
    dataset_dir = os.path.join(current_dir, 'dataset')

train_dir = os.path.join(dataset_dir, 'train', 'train')
valid_dir = os.path.join(dataset_dir, 'valid')

if os.path.exists(train_dir) and os.path.exists(valid_dir):
    print(f"✓ Dataset found: {len(os.listdir(train_dir))} classes")
    print(f"Running on: {'Google Colab' if IN_COLAB else 'Local'}")
else:
    print(f"✗ Dataset not found!")
    print(f"Expected train path: {train_dir}")
    print(f"Expected valid path: {valid_dir}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using {device}")

using cpu


In [5]:
# ImageNet stats buat normalisasi
stats = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

# transform buat training - pake augmentasi biar model lebih robust
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),  # flip kiri-kanan
    transforms.RandomRotation(15),      # rotasi max 15 derajat
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),  # variasi warna
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

# transform buat validasi/test - gak pake augmentasi
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])
# custom dataset class buat apply transform ke subset
class DatasetFromSubset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
    
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    
    def __len__(self):
        return len(self.subset)

## Split Dataset

In [6]:
# load dataset
full_dataset = datasets.ImageFolder(root=train_dir)
class_names = full_dataset.classes

# split jadi 60-20-20 untuk train-val-test
total_size = len(full_dataset)
train_size = int(0.6 * total_size)
val_size = int(0.2 * total_size)
test_size = total_size - train_size - val_size

print(f"dataset: {total_size} samples, {len(class_names)} classes | train/val/test: {train_size}/{val_size}/{test_size}")

# split dataset
train_subset, val_subset, test_subset = random_split(
    full_dataset, 
    [train_size, val_size, test_size]
)

# apply transformasi ke masing-masing subset
train_data = DatasetFromSubset(train_subset, transform=train_transform)
valid_data = DatasetFromSubset(val_subset, transform=val_test_transform)
test_data = DatasetFromSubset(test_subset, transform=val_test_transform)

FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset/train/train'

In [ ]:
# setup dataloader
BATCH_SIZE = 32
NUM_WORKERS = 0 if IN_COLAB else 2  # Colab kadang bermasalah dengan num_workers > 0

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Batch size: {BATCH_SIZE}, Workers: {NUM_WORKERS}")

## Model 1: CNN Standar

In [ ]:
# bikin CNN sederhana dengan 4 conv blocks
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        
        # feature extractor
        self.features = nn.Sequential(
            # layer 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # layer 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # layer 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # layer 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        
        # classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(),
            nn.Dropout(0.3),  # dropout buat regularisasi
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

cnn_model = SimpleCNN(num_classes=len(class_names)).to(device)
print(f"CNN: {sum(p.numel() for p in cnn_model.parameters()):,} params")

CNN: 26,095,969 params


## Model 2: ResNet50

In [ ]:
# load ResNet50 pretrained
def create_resnet50(num_classes):
    model = models.resnet50(pretrained=True)
    
    # freeze semua layer kecuali classifier
    for param in model.parameters():
        param.requires_grad = False
    
    # ganti fc layer terakhir
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(num_features, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )
    
    return model

resnet_model = create_resnet50(num_classes=len(class_names)).to(device)
print(f"ResNet50: {sum(p.numel() for p in resnet_model.parameters() if p.requires_grad):,} trainable")

C:\Users\LENOVO\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\LENOVO\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet50: 1,066,017 trainable


## Model 3: EfficientNet-B0

In [ ]:
# load EfficientNet pretrained
def create_efficientnet_b0(num_classes):
    model = models.efficientnet_b0(pretrained=True)
    
    # freeze base model
    for param in model.parameters():
        param.requires_grad = False
    
    # custom classifier
    num_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(num_features, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )
    
    return model

efficientnet_model = create_efficientnet_b0(num_classes=len(class_names)).to(device)
print(f"EfficientNet: {sum(p.numel() for p in efficientnet_model.parameters() if p.requires_grad):,} trainable")

C:\Users\LENOVO\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


EfficientNet: 672,801 trainable


## Training

In [ ]:
def train_model(model, model_name, train_loader, valid_loader, num_epochs=10, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=0.5)
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'epoch_time': []}
    best_val_acc = 0.0
    model_path = f'best_{model_name.lower().replace(" ", "_")}.pth'
    
    print(f"\n[{model_name}] starting...")
    
    for epoch in range(num_epochs):
        start_time = time.time()
        
        # training
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        for images, labels in tqdm(train_loader, desc=f"ep {epoch+1}/{num_epochs}", leave=False):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
        
        avg_train_loss = train_loss / len(train_loader)
        train_acc = 100 * train_correct / train_total
        
        # validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        
        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        avg_val_loss = val_loss / len(valid_loader)
        val_acc = 100 * val_correct / val_total
        epoch_time = time.time() - start_time
        
        scheduler.step(val_acc)
        
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)
        history['epoch_time'].append(epoch_time)
        
        print(f"ep {epoch+1}: loss {avg_train_loss:.3f}/{avg_val_loss:.3f}, acc {train_acc:.1f}/{val_acc:.1f}%", end="")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), model_path)
            print(" ✓ saved")
        else:
            print()
    
    print(f"done! best: {best_val_acc:.2f}%\n")
    return history, model_path

## Mulai Training

In [ ]:
# training CNN dulu
cnn_history, cnn_path = train_model(
    cnn_model, "CNN Standar", 
    train_loader, valid_loader, 
    num_epochs=10, lr=0.001
)


[CNN Standar] starting...


ep 1/10:   0%|          | 0/316 [00:00<?, ?it/s]

In [ ]:
# training ResNet50
resnet_history, resnet_path = train_model(
    resnet_model, "ResNet50",
    train_loader, valid_loader,
    num_epochs=10, lr=0.001
)

In [ ]:
# training EfficientNet
efficientnet_history, efficientnet_path = train_model(
    efficientnet_model, "EfficientNet-B0",
    train_loader, valid_loader,
    num_epochs=10, lr=0.001
)

## Backup Model ke Google Drive (opsional)

In [ ]:
# Backup trained models ke Google Drive biar gak ilang
if IN_COLAB:
    import shutil
    drive_backup_path = '/content/drive/MyDrive/lab_models'
    
    # Create directory if not exists
    os.makedirs(drive_backup_path, exist_ok=True)
    
    # Copy model files
    model_files = [cnn_path, resnet_path, efficientnet_path]
    for model_file in model_files:
        if os.path.exists(model_file):
            shutil.copy(model_file, drive_backup_path)
            print(f"✓ Backed up: {model_file}")
    
    print(f"\nAll models saved to: {drive_backup_path}")
else:
    print("Running locally - models already saved to current directory")

## Evaluasi Model

In [ ]:
def evaluate_model(model, model_path, test_loader, class_names, model_name):
    model.load_state_dict(torch.load(model_path))
    model.eval()
    
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f"eval {model_name}", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=class_names, digits=4, zero_division=0)
    
    print(f"\n{model_name} -> acc: {accuracy*100:.2f}%, prec: {precision:.3f}, recall: {recall:.3f}, f1: {f1:.3f}")
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'confusion_matrix': cm,
        'classification_report': report,
        'predictions': all_preds,
        'labels': all_labels
    }

## Testing

In [ ]:
cnn_results = evaluate_model(cnn_model, cnn_path, test_loader, class_names, "CNN")

In [ ]:
resnet_results = evaluate_model(resnet_model, resnet_path, test_loader, class_names, "ResNet50")

In [ ]:
efficientnet_results = evaluate_model(efficientnet_model, efficientnet_path, test_loader, class_names, "EfficientNet")

## Visualisasi Confusion Matrix

In [ ]:
def plot_confusion_matrices_3models(cnn_cm, resnet_cm, efficientnet_cm, class_names):
    fig, axes = plt.subplots(1, 3, figsize=(24, 7))
    
    # CNN Standar
    sns.heatmap(cnn_cm, annot=False, fmt='d', cmap='Greens', 
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[0], cbar_kws={'label': 'Count'})
    axes[0].set_title('CNN Standar - Confusion Matrix', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Predicted', fontsize=11)
    axes[0].set_ylabel('True', fontsize=11)
    axes[0].tick_params(axis='both', labelsize=7)
    
    # ResNet50
    sns.heatmap(resnet_cm, annot=False, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[1], cbar_kws={'label': 'Count'})
    axes[1].set_title('ResNet50 - Confusion Matrix', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Predicted', fontsize=11)
    axes[1].set_ylabel('True', fontsize=11)
    axes[1].tick_params(axis='both', labelsize=7)
    
    # EfficientNet-B0
    sns.heatmap(efficientnet_cm, annot=False, fmt='d', cmap='Reds',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[2], cbar_kws={'label': 'Count'})
    axes[2].set_title('EfficientNet-B0 - Confusion Matrix', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Predicted', fontsize=11)
    axes[2].set_ylabel('True', fontsize=11)
    axes[2].tick_params(axis='both', labelsize=7)
    
    plt.tight_layout()
    plt.savefig('confusion_matrices_all_models.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_confusion_matrices_3models(
    cnn_results['confusion_matrix'],
    resnet_results['confusion_matrix'],
    efficientnet_results['confusion_matrix'],
    class_names
)

## Perbandingan Performa

In [ ]:
def plot_metrics_comparison_3models(cnn_res, resnet_res, efficientnet_res):
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    cnn_values = [cnn_res['accuracy'], cnn_res['precision'], cnn_res['recall'], cnn_res['f1_score']]
    resnet_values = [resnet_res['accuracy'], resnet_res['precision'], resnet_res['recall'], resnet_res['f1_score']]
    efficientnet_values = [efficientnet_res['accuracy'], efficientnet_res['precision'], efficientnet_res['recall'], efficientnet_res['f1_score']]
    
    x = np.arange(len(metrics))
    width = 0.25
    
    fig, ax = plt.subplots(figsize=(14, 7))
    bars1 = ax.bar(x - width, cnn_values, width, label='CNN Standar', 
                   color='mediumseagreen', edgecolor='black')
    bars2 = ax.bar(x, resnet_values, width, label='ResNet50',
                   color='steelblue', edgecolor='black')
    bars3 = ax.bar(x + width, efficientnet_values, width, label='EfficientNet-B0',
                   color='coral', edgecolor='black')
    
    ax.set_xlabel('Metrics', fontsize=13, fontweight='bold')
    ax.set_ylabel('Score', fontsize=13, fontweight='bold')
    ax.set_title('Perbandingan Performa: CNN vs ResNet50 vs EfficientNet-B0', 
                 fontsize=15, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics, fontsize=12)
    ax.legend(fontsize=12)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0, 1.1])
    
    # Add value labels on bars
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.4f}',
                   ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('metrics_comparison_all_models.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_metrics_comparison_3models(cnn_results, resnet_results, efficientnet_results)

## Tabel Summary

In [ ]:
def create_summary_table_3models(cnn_res, resnet_res, efficientnet_res, cnn_hist, resnet_hist, efficientnet_hist):
    data = {
        'Model': ['CNN', 'ResNet50', 'EfficientNet'],
        'Acc (%)': [f"{cnn_res['accuracy']*100:.2f}", f"{resnet_res['accuracy']*100:.2f}", f"{efficientnet_res['accuracy']*100:.2f}"],
        'Precision': [f"{cnn_res['precision']:.3f}", f"{resnet_res['precision']:.3f}", f"{efficientnet_res['precision']:.3f}"],
        'Recall': [f"{cnn_res['recall']:.3f}", f"{resnet_res['recall']:.3f}", f"{efficientnet_res['recall']:.3f}"],
        'F1': [f"{cnn_res['f1_score']:.3f}", f"{resnet_res['f1_score']:.3f}", f"{efficientnet_res['f1_score']:.3f}"],
        'Best Val (%)': [f"{max(cnn_hist['val_acc']):.1f}", f"{max(resnet_hist['val_acc']):.1f}", f"{max(efficientnet_hist['val_acc']):.1f}"]
    }
    
    df = pd.DataFrame(data)
    print("\n" + df.to_string(index=False))
    df.to_csv('comparison.csv', index=False)
    print("\nsaved to comparison.csv")
    return df

summary_df = create_summary_table_3models(cnn_results, resnet_results, efficientnet_results, cnn_history, resnet_history, efficientnet_history)

## Training History

In [ ]:
def plot_training_history_3models(cnn_hist, resnet_hist, efficientnet_hist):
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    epochs_cnn = range(1, len(cnn_hist['train_loss']) + 1)
    epochs_resnet = range(1, len(resnet_hist['train_loss']) + 1)
    epochs_efficient = range(1, len(efficientnet_hist['train_loss']) + 1)
    
    # Training Loss
    axes[0, 0].plot(epochs_cnn, cnn_hist['train_loss'], 'g-', label='CNN Standar', linewidth=2)
    axes[0, 0].plot(epochs_resnet, resnet_hist['train_loss'], 'b-', label='ResNet50', linewidth=2)
    axes[0, 0].plot(epochs_efficient, efficientnet_hist['train_loss'], 'r-', label='EfficientNet-B0', linewidth=2)
    axes[0, 0].set_title('Training Loss Comparison', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Validation Loss
    axes[0, 1].plot(epochs_cnn, cnn_hist['val_loss'], 'g-', label='CNN Standar', linewidth=2)
    axes[0, 1].plot(epochs_resnet, resnet_hist['val_loss'], 'b-', label='ResNet50', linewidth=2)
    axes[0, 1].plot(epochs_efficient, efficientnet_hist['val_loss'], 'r-', label='EfficientNet-B0', linewidth=2)
    axes[0, 1].set_title('Validation Loss Comparison', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Training Accuracy
    axes[1, 0].plot(epochs_cnn, cnn_hist['train_acc'], 'g-', label='CNN Standar', linewidth=2)
    axes[1, 0].plot(epochs_resnet, resnet_hist['train_acc'], 'b-', label='ResNet50', linewidth=2)
    axes[1, 0].plot(epochs_efficient, efficientnet_hist['train_acc'], 'r-', label='EfficientNet-B0', linewidth=2)
    axes[1, 0].set_title('Training Accuracy Comparison', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Accuracy (%)')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Validation Accuracy
    axes[1, 1].plot(epochs_cnn, cnn_hist['val_acc'], 'g-', label='CNN Standar', linewidth=2)
    axes[1, 1].plot(epochs_resnet, resnet_hist['val_acc'], 'b-', label='ResNet50', linewidth=2)
    axes[1, 1].plot(epochs_efficient, efficientnet_hist['val_acc'], 'r-', label='EfficientNet-B0', linewidth=2)
    axes[1, 1].set_title('Validation Accuracy Comparison', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Accuracy (%)')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_history_all_models.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_training_history_3models(cnn_history, resnet_history, efficientnet_history)

## Analisis Hasil

In [ ]:
def generate_analysis_3models(cnn_res, resnet_res, efficientnet_res):
    models = [("CNN", cnn_res), ("ResNet50", resnet_res), ("EfficientNet", efficientnet_res)]
    models.sort(key=lambda x: x[1]['accuracy'], reverse=True)
    
    print("\nRanking:")
    for i, (name, res) in enumerate(models, 1):
        print(f"{i}. {name}: {res['accuracy']*100:.2f}%")
    
    best = models[0][0]
    print(f"\nWinner: {best}")
    
    print(f"\nMetrics detail:")
    print(f"CNN       : acc {cnn_res['accuracy']:.3f}, prec {cnn_res['precision']:.3f}, recall {cnn_res['recall']:.3f}, f1 {cnn_res['f1_score']:.3f}")
    print(f"ResNet50  : acc {resnet_res['accuracy']:.3f}, prec {resnet_res['precision']:.3f}, recall {resnet_res['recall']:.3f}, f1 {resnet_res['f1_score']:.3f}")
    print(f"EfficientNet: acc {efficientnet_res['accuracy']:.3f}, prec {efficientnet_res['precision']:.3f}, recall {efficientnet_res['recall']:.3f}, f1 {efficientnet_res['f1_score']:.3f}")
    
    print("\nKesimpulan:")
    if best == "CNN":
        print("- CNN simple cukup bagus untuk dataset ini")
        print("- model kompleks kayaknya overkill")
    elif best == "ResNet50":
        print("- ResNet skip connection works well")
        print("- pretrained weight ngebantu banget")
    else:
        print("- EfficientNet paling balance")
        print("- efficiency + accuracy dapet")

generate_analysis_3models(cnn_results, resnet_results, efficientnet_results)